
# Gold Layer – Dublin Bikes

Create business-ready datasets for reporting and analytics.


## Gold Table 1 – Station Utilisation

**Business Question:** Which Dublin Bikes stations have the highest average bike availability?

This Gold table provides the average number of bikes available and the average number of empty docks for each station. The dataset is intended for reporting and dashboarding.

In [0]:
-- Create the Gold table
CREATE OR REPLACE TABLE urban_mobility.gold.station_utilisation AS

SELECT
    station_id,
    name AS station_name,
    capacity,
    ROUND(AVG(num_bikes_available), 2) AS average_bikes_available,
    ROUND(AVG(capacity - num_bikes_available), 2) AS average_empty_docks
FROM urban_mobility.silver.stations
GROUP BY
    station_id,
    name,
    capacity;

In [0]:
-- Verify the Gold table
SELECT *
FROM urban_mobility.gold.station_utilisation
ORDER BY average_bikes_available DESC
LIMIT 5 ;


## Gold Table 2 – Station Occupancy

**Business Question:** Which Dublin Bikes stations have the highest average occupancy?

This Gold table calculates the average occupancy percentage for each station by comparing the average number of bikes available with the station capacity. The dataset helps identify stations that are consistently full or underutilised and supports operational reporting and dashboarding.

In [0]:
-- Create the Gold table
CREATE OR REPLACE TABLE urban_mobility.gold.station_occupancy AS

SELECT
    station_id,
    name AS station_name,
    ROUND((1 - AVG(num_docks_available / capacity)) * 100, 2) AS average_occupancy_percentage
FROM urban_mobility.silver.stations
GROUP BY
    station_id,
    name;


## Gold Table 3 – Peak Hour Station Rankings

**Business Question:** Which Dublin Bikes stations have the highest average occupancy during each hour of the day?

This Gold table analyses average station occupancy by hour and ranks stations based on their occupancy percentage. The dataset helps identify the busiest stations throughout the day, supporting operational planning, bike redistribution, and demand analysis.

In [0]:
SELECT * FROM urban_mobility.silver.stations LIMIT 3 ;

In [0]:
WITH hourly_occupancy AS (

    SELECT
        station_id,
        name,
        HOUR(last_reported) AS report_hour,
        ROUND((1 - AVG(num_docks_available / capacity)) * 100, 2) AS average_occupancy_percentage
    FROM urban_mobility.silver.stations
    GROUP BY
        station_id,
        name,
        HOUR(last_reported)

),

hourly_rankings AS (

    SELECT
        name,
        report_hour,
        average_occupancy_percentage,
        RANK() OVER (
            PARTITION BY report_hour
            ORDER BY average_occupancy_percentage DESC
        ) AS station_rank
    FROM hourly_occupancy

)

SELECT *
FROM hourly_rankings
WHERE station_rank = 1;


## Gold Table 4 – Station Status Classification

**Business Question:** Which Dublin Bikes stations most frequently operate in a critical status throughout the day?

This Gold table classifies each station reading into operational status categories based on occupancy percentage. Stations are labelled as **Nearly Full**, **Nearly Empty**, or **Normal**, allowing operators to identify stations that regularly require bike redistribution or additional capacity. The results support operational monitoring, resource planning, and service reliability.

In [0]:
WITH occupancy AS (

    SELECT
        name,
        ROUND((1 - (num_docks_available / capacity)) * 100, 2) AS occupancy_percentage
    FROM urban_mobility.silver.stations

),

station_status AS (

    SELECT
        name,
        occupancy_percentage,
        CASE
            WHEN occupancy_percentage > 90 THEN 'Highly utilized'
            WHEN occupancy_percentage < 10 THEN 'Nearly empty'
            ELSE 'Moderately utilized'
        END AS station_status
    FROM occupancy

)

SELECT
    name,
    station_status,
    COUNT(*) AS total_occurrences
FROM station_status
GROUP BY
    name,
    station_status
ORDER BY
    total_occurrences DESC;


## Gold Table 5 – Bike Availability Change Detection

**Business Question:** Which Dublin Bikes stations experience the largest changes in bike availability between consecutive updates?

This Gold table analyses changes in bike availability over time by comparing each station's current bike count with its previous recorded value. The analysis helps identify sudden increases or decreases in availability, supporting demand monitoring, bike redistribution planning, and operational decision making.

In [0]:
WITH bike_changes AS (

    SELECT
        station_id,
        name,
        last_reported,
        num_bikes_available,
        LAG(num_bikes_available) OVER (
            PARTITION BY station_id
            ORDER BY last_reported
        ) AS previous_bikes
    FROM urban_mobility.silver.stations

),

availability_changes AS (

    SELECT
        station_id,
        name,
        last_reported,
        num_bikes_available,
        previous_bikes,
        num_bikes_available - previous_bikes AS bike_change
    FROM bike_changes

)

SELECT
    station_id,
    name,
    last_reported,
    num_bikes_available,
    previous_bikes,
    bike_change
FROM availability_changes
ORDER BY ABS(bike_change) DESC;